<a href="https://colab.research.google.com/github/Stasia2/hate_speech_detection/blob/main/Copy_of_Welcome_To_Colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# New Section

In [1]:
!pip install nltk
!pip install streamlit

import pandas as pd
import numpy as np
import random
import nltk

nltk.download('punkt')
nltk.download('stopwords')
nltk.download('punkt_tab')

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.1/9.1 MB 70.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.9/6.9 MB 95.9 MB/s eta 0:00:00


[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.
[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.


True

In [2]:
import pandas as pd
import random

# Base words
subjects = [
    "you", "ndi a", "that person", "those people", "this guy",
    "that girl", "ndi obodo a", "that man", "this woman"
]

hate_words = [
    "stupid", "useless", "foolish", "wicked", "bad",
    "crazy", "senseless", "idiotic", "annoying", "dumb"
]

non_hate_words = [
    "good", "kind", "nice", "amazing", "brilliant",
    "wonderful", "smart", "lovely", "great", "helpful"
]

verbs = [
    "are", "is", "seems", "looks", "feels"
]

extras = [
    "very", "so", "really", "too much", "quite", "extremely"
]

endings = [
    "", "today", "honestly", "in this life", "sometimes"
]

# Generate dataset
data = []

for _ in range(2500):
    sentence = f"{random.choice(subjects)} {random.choice(verbs)} {random.choice(extras)} {random.choice(hate_words)} {random.choice(endings)}"
    data.append((sentence, "hate"))

for _ in range(2500):
    sentence = f"{random.choice(subjects)} {random.choice(verbs)} {random.choice(extras)} {random.choice(non_hate_words)} {random.choice(endings)}"
    data.append((sentence, "not_hate"))

# Create DataFrame
df = pd.DataFrame(data, columns=["text", "label"])

df.head()

,text,label
0,that person is really annoying sometimes,hate
1,you are very useless today,hate
2,that person seems quite senseless sometimes,hate
3,that girl feels too much foolish honestly,hate
4,that girl seems very dumb today,hate


In [3]:
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords

stop_words = set(stopwords.words('english'))

def clean_text(text):
    words = word_tokenize(text.lower())
    words = [w for w in words if w.isalpha()]
    words = [w for w in words if w not in stop_words]
    return " ".join(words)

df["clean"] = df["text"].apply(clean_text)

df.head()

,text,label,clean
0,that person is really annoying sometimes,hate,person really annoying sometimes
1,you are very useless today,hate,useless today
2,that person seems quite senseless sometimes,hate,person seems quite senseless sometimes
3,that girl feels too much foolish honestly,hate,girl feels much foolish honestly
4,that girl seems very dumb today,hate,girl seems dumb today


In [4]:
from sklearn.feature_extraction.text import TfidfVectorizer

vectorizer = TfidfVectorizer()

X = vectorizer.fit_transform(df["clean"])
y = df["label"]

In [5]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

In [6]:
from sklearn.naive_bayes import MultinomialNB
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier

nb = MultinomialNB().fit(X_train, y_train)
lr = LogisticRegression().fit(X_train, y_train)
svm = SVC().fit(X_train, y_train)
dt = DecisionTreeClassifier().fit(X_train, y_train)

print("All models trained")

All models trained


In [7]:
from sklearn.metrics import accuracy_score

print("Naive Bayes:", accuracy_score(y_test, nb.predict(X_test)))
print("Logistic Regression:", accuracy_score(y_test, lr.predict(X_test)))
print("SVM:", accuracy_score(y_test, svm.predict(X_test)))
print("Decision Tree:", accuracy_score(y_test, dt.predict(X_test)))

Naive Bayes: 1.0
Logistic Regression: 1.0
SVM: 1.0
Decision Tree: 1.0


In [8]:
test_text = ["ndi a bu useless people"]

vec = vectorizer.transform(test_text)

print("Prediction:", svm.predict(vec))

Prediction: ['hate']


In [10]:
%%writefile app.py
import streamlit as st
import pandas as pd
import random

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.svm import SVC

# --- DATA GENERATION ---
subjects = ["you", "ndi a", "that person", "those people"]
hate_words = ["stupid", "useless", "foolish", "wicked"]
non_hate_words = ["good", "kind", "nice", "amazing"]
verbs = ["are", "is"]
extras = ["very", "so", "really"]

data = []

for _ in range(2500):
    sentence = f"{random.choice(subjects)} {random.choice(verbs)} {random.choice(extras)} {random.choice(hate_words)}"
    data.append((sentence, "hate"))

for _ in range(2500):
    sentence = f"{random.choice(subjects)} {random.choice(verbs)} {random.choice(extras)} {random.choice(non_hate_words)}"
    data.append((sentence, "not_hate"))

df = pd.DataFrame(data, columns=["text", "label"])

# --- CLEAN ---
df["clean"] = df["text"].str.lower()

# --- VECTORIZE ---
vectorizer = TfidfVectorizer()
X = vectorizer.fit_transform(df["clean"])
y = df["label"]

# --- TRAIN MODEL ---
svm = SVC()
svm.fit(X, y)

# --- GUI ---
st.title("Hate Speech Detection System")

text = st.text_input("Enter text")

if st.button("Check"):
    if text == "":
        st.write("Please enter text")
    else:
        vec = vectorizer.transform([text.lower()])
        result = svm.predict(vec)
        st.write("Prediction:", result[0])

Overwriting app.py


In [11]:
!npm install -g localtunnel

⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹
added 22 packages in 3s
⠸
⠸3 packages are looking for funding
⠸  run `npm fund` for details
⠸

In [21]:
%%writefile app.py
import streamlit as st
import pandas as pd
import random

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.svm import SVC

# --- DATA GENERATION ---
subjects = ["you", "ndi a", "that person", "those people"]
hate_words = ["stupid", "useless", "foolish", "wicked"]
non_hate_words = ["good", "kind", "nice", "amazing"]
verbs = ["are", "is"]
extras = ["very", "so", "really"]

data = []

for _ in range(2500):
    sentence = f"{random.choice(subjects)} {random.choice(verbs)} {random.choice(extras)} {random.choice(hate_words)}"
    data.append((sentence, "hate"))

for _ in range(2500):
    sentence = f"{random.choice(subjects)} {random.choice(verbs)} {random.choice(extras)} {random.choice(non_hate_words)}"
    data.append((sentence, "not_hate"))

df = pd.DataFrame(data, columns=["text", "label"])

# --- CLEAN ---
df["clean"] = df["text"].str.lower()

# --- VECTORIZE ---
vectorizer = TfidfVectorizer()
X = vectorizer.fit_transform(df["clean"])
y = df["label"]

# --- TRAIN MODEL ---
svm = SVC()
svm.fit(X, y)

# --- GUI ---
st.title("Hate Speech Detection System")

text = st.text_input("Enter text")

if st.button("Check"):
    if text == "":
        st.write("Please enter text")
    else:
        vec = vectorizer.transform([text.lower()])
        result = svm.predict(vec)
        st.write("Prediction:", result[0])

Overwriting app.py


In [22]:
%%writefile requirements.txt
streamlit
pandas
scikit-learn

Writing requirements.txt


In [23]:
!ls

app.py	requirements.txt  sample_data


In [24]:
from google.colab import files

files.download('app.py')
files.download('requirements.txt')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>